# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duashakeel0/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Which pages should a content editor review first for refresh, given limited weekly capacity?**

This supports one decision: how an editor allocates scarce review time across a large page inventory. Full framing (decision, action, cost of a wrong call, why ML helps) lives in `work/capstone_report.md` Section 1 -- this notebook verifies the numbers that report cites, against the metrics files already committed across Weeks 1-7, rather than re-deriving them from scratch.

In [1]:
import json

def load(name):
    with open(f"../outputs/{name}") as f:
        return json.load(f)

m05 = load("w05_model_metrics.json")
m06 = load("w06_validation_audit_metrics.json")
m07 = load("w07_action_playbook_metrics.json")

print("Lane: Refresh / Content Opportunity Scoring")
print("Decision point:", m05["data"]["decision_point"])
print("Feature window:", m05["data"]["feature_window"], " | Target window:", m05["data"]["target_window"])

Lane: Refresh / Content Opportunity Scoring
Decision point: 2026-03-15
Feature window: 2026-03-01 to 2026-03-15  | Target window: 2026-03-16 to 2026-03-31


## 2. Data

FlyRank internship warehouse (Hugging Face, gated), `fact_content_daily_performance/month=2026-03` (9.84M rows) + `dim_content` (519,606 rows). Full exclusion list and reasons in `work/capstone_report.md` Section 2 -- the 81% point-in-time drop is the headline number to verify below.

In [2]:
bug = m05["bugs_found_and_fixed"][0]
assert bug["name"] == "point_in_time_leakage"
print(f"Rows before point-in-time fix: {bug['rows_before']:,}")
print(f"Rows dropped (future-dated dim_content): {bug['rows_dropped']:,} ({bug['pct_dropped']}%)")
print(f"Rows after fix (final modeling population): {bug['rows_after']:,}")
print(f"\nExcluded features: {m05['excluded_from_features']}")

Rows before point-in-time fix: 92,247
Rows dropped (future-dated dim_content): 74,624 (80.9%)
Rows after fix (final modeling population): 17,623

Excluded features: ['impressions_h2 (defines the label)', 'client_hash_id/content_hash_id (context only)']


## 3. Methodology

Logistic Regression then Random Forest, 8 features (all knowable by the decision point), client-holdout split. Baseline: the rescaled `stale_but_visible` rule. Full reasoning in `work/capstone_report.md` Sections 3-4.

In [3]:
print("Features:", m05["features"])
print("Label:", m05["label"]["definition"], " | min volume:", m05["label"]["min_volume_filter"])
print("Split strategy:", m05["split"]["strategy"], "| train/test clients:",
      m05["split"]["train_clients"], "/", m05["split"]["test_clients"],
      "| client overlap:", m05["split"]["client_overlap"])

bug2 = m05["bugs_found_and_fixed"][1]
print(f"\nReproducibility bug found: {bug2['name']}")
print(f"  {bug2['description']}")
print(f"  Verification: {bug2['verification']}")

Features: ['impressions_h1', 'clicks_h1', 'avg_position_h1', 'sessions_h1', 'engaged_sessions_h1', 'days_since_last_update', 'content_age_days', 'word_count']
Label: impressions_h2 < impressions_h1 (Mar 16-31 impressions < Mar 1-15 impressions)  | min volume: impressions_h1 >= 50
Split strategy: client_holdout | train/test clients: 22 / 6 | client overlap: 0

Reproducibility bug found: nondeterministic_remote_row_order
  DuckDB does not guarantee row order on a remote parquet scan without ORDER BY, so .unique() and argsort tie-breaking were not actually deterministic despite a fixed numpy seed -- reruns produced different comparison tables.
  Verification: 3 independent full reruns (A/B/C) produced identical results


## 4. Results (vs baseline)

The honest table, verified from committed metrics, plus the two charts the paper embeds: model vs. baseline, and the before/after split-honesty proof from Week 6.

In [4]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

os.makedirs("../outputs/figures", exist_ok=True)

# Theme-safe styling: transparent background (so it sits cleanly in a light OR
# dark card on the deployed page), mid-luminance grid/text that reads on both.
mpl.rcParams.update({
    "figure.facecolor": "none", "axes.facecolor": "none", "savefig.facecolor": "none",
    "text.color": "#7a7d74", "axes.labelcolor": "#7a7d74",
    "xtick.color": "#7a7d74", "ytick.color": "#7a7d74",
    "axes.edgecolor": "#7a7d74", "grid.color": "#7a7d74", "grid.alpha": 0.15,
    "font.family": "serif", "font.size": 11.5,
})
HONEST, DISHONEST, NEUTRAL, ACCENT2 = "#3a9b7d", "#b8532e", "#9a9a90", "#6b8fa3"

comp = m05["comparison"]
print("Test set:", m05["split"]["test_rows"], "rows |", m05["split"]["test_clients"], "held-out clients")
print("Base rate:", m05["label"]["base_rate_test_set"])
for method, vals in comp.items():
    print(f"  {method}: P@20={vals['precision_at_20']}  P@50={vals['precision_at_50']}")

# --- Figure A: model vs baseline at P@50 (no honest/dishonest color semantic here --
# that's reserved for Figure B's split comparison; these three are just three methods) ---
methods = ["Baseline\n(never fires)", "Logistic\nRegression", "Random\nForest"]
p50 = [0, comp["logistic_regression"]["precision_at_50"], comp["random_forest"]["precision_at_50"]]
colors_a = [NEUTRAL, ACCENT2, HONEST]

fig, ax = plt.subplots(figsize=(6.4, 4.2))
bars = ax.bar(methods, p50, color=colors_a, width=0.55, zorder=3)
ax.axhline(m05["label"]["base_rate_test_set"], color="#7a7d74", linestyle=(0, (4, 3)), linewidth=1.2, zorder=2)
ax.text(2.35, m05["label"]["base_rate_test_set"] + 0.015, "base rate", fontsize=10, ha="right")
ax.set_ylabel("Precision@50")
ax.set_ylim(0, max(p50) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", zorder=0)
for b, v in zip(bars, p50):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}" if v else "N/A", ha="center", fontsize=12, fontweight="bold", color="#3d403a")
plt.tight_layout()
fig.savefig("../outputs/figures/capstone_model_vs_baseline.png", dpi=180, transparent=True)
plt.close(fig)

# --- Figure B: before/after split honesty (this is where honest/dishonest colors apply) ---
before = m06["before_after_split_comparison"]["before_random_row_split"]["precision_at_50"]
after = m06["before_after_split_comparison"]["after_client_holdout_split"]["precision_at_50"]
fig, ax = plt.subplots(figsize=(6.4, 4.2))
bars = ax.bar(["Naive random split\n(dishonest)", "Client-holdout split\n(honest)"], [before, after],
              color=[DISHONEST, HONEST], width=0.5, zorder=3)
ax.set_ylabel("Precision@50")
ax.set_ylim(0, max(before, after) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", zorder=0)
for b, v in zip(bars, [before, after]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=13, fontweight="bold", color="#3d403a")
ax.text(0.5, (before + after) / 2, f"{before - after:+.2f} gap", ha="center", fontsize=10, color="#7a7d74")
plt.tight_layout()
fig.savefig("../outputs/figures/capstone_split_honesty.png", dpi=180, transparent=True)
plt.close(fig)

print("\nWrote capstone_model_vs_baseline.png and capstone_split_honesty.png (transparent, theme-safe)")

Test set: 1979 rows | 6 held-out clients
Base rate: 0.356
  baseline_rescaled_rule: P@20=None  P@50=None
  logistic_regression: P@20=0.65  P@50=0.46
  random_forest: P@20=0.6  P@50=0.64



Wrote capstone_model_vs_baseline.png and capstone_split_honesty.png (transparent, theme-safe)


## 5. Limitations

- Single month (March 2026), single client-holdout split; only 6 test clients (Week 6).
- Population selection gap: pages edited between March 15 and July are excluded entirely (the point-in-time fix's side effect) -- may under-represent decliners editors already caught.
- Staleness (`days_since_last_update`) is a consistently weak signal across two independent checks (Week 4 starter CSV: MIXED; Week 5/6 warehouse: lowest feature importance) -- named directly in Section 6 of the report, not buried.
- Not causal: cross-sectional data, no experiment. A `prioritize_review` tag means "worth a look," never "refreshing this will fix it."
- Small high-confidence tier: only 11 of 1,979 rows reach `prioritize_review` -- most of the population sits in the ambiguous `monitor` tier, reported as-is.

In [5]:
gap = m06["before_after_split_comparison"]["gap"]
print(f"Random-vs-honest split gap (memorization evidence): {gap:+.3f}")
print(f"Leakage re-test: honest AUC {m06['leakage_audit']['live_retest_on_week5_final_features']['honest_auc_8_features']}"
      f" vs leaky AUC {m06['leakage_audit']['live_retest_on_week5_final_features']['leaky_auc_8_features_plus_impressions_h2']}")
print(f"New limitation found: {m06['leakage_audit']['new_limitation_found'][:120]}...")

Random-vs-honest split gap (memorization evidence): +0.200
Leakage re-test: honest AUC 0.685 vs leaky AUC 0.936
New limitation found: Week 5's leakage fix (dropping rows with future-dated content_updated_date) means the modeling population is, by constru...


## 6. Ranked recommendations

The action playbook output (Week 7), built only from the validated test population. Full human-review checklist and no-go list in `work/capstone_report.md` Section 7.

In [6]:
q = m07["queue"]
print("Action tier breakdown (n =", q["total_rows"], "):")
for tier, n in q["action_counts"].items():
    print(f"  {tier}: {n}")
print("\nReason codes among prioritize_review:", q["reason_codes_among_prioritize_review"])
print("\nNo-go list:")
for item in m07["no_go_list"]:
    print(" -", item)

Action tier breakdown (n = 1979 ):
  prioritize_review: 11
  monitor: 1763
  no_action_needed: 205

Reason codes among prioritize_review: {'high_exposure_declining_risk': 9, 'poor_position_declining_risk': 2}

No-go list:
 - never auto-publish a content change from this queue
 - never treat prioritize_review as proof of decline (precision 0.64, ~1/3 flagged rows are false positives)
 - never use this queue to justify removing a page or deprioritizing a client
 - never skip the human-review checklist because the model score is high


## 7. Artifacts the paper embeds

In [7]:
import os
figs = sorted(os.listdir("../outputs/figures"))
print("Figures ready for the deployed paper:")
for f in figs:
    print(" -", f)

Figures ready for the deployed paper:
 - capstone_model_vs_baseline.png
 - capstone_split_honesty.png
 - w07_action_tier_breakdown.png


## ML-12 — Demo outline, social cut, employer summary

**5-minute demo outline** (live walkthrough order):
1. (30s) The question: which pages should an editor review first, given limited time?
2. (60s) Show the baseline rule -- and that it never fires on the held-out test set at all.
3. (60s) Show the model-vs-baseline chart: Random Forest reaches P@50=0.64 against a base rate of 0.356.
4. (90s) The split-honesty proof: same data, same model, naive split scores 0.84 vs the honest 0.64 -- a 20-point gap that's really about memorization, not skill. This is the moment that makes the whole project's rigor visible.
5. (60s) The ranked queue + reason codes + the no-go list -- what an editor actually gets, and what they must never do with it.
6. (20s) The honest limitation: staleness, the lane's own namesake signal, turned out to be the weakest feature in the model -- twice, independently.

**Social-post cut** (one paragraph, shareable):
> I spent 8 weeks building a page-refresh priority model on real Search Console + Analytics data. The most useful thing I found wasn't the model -- it was a bug. My "validated" split looked 20 points better than it should have, because rows from the same client were leaking between train and test. Fixing it dropped my headline number from a shiny 0.84 to an honest 0.64. That 0.64 is real. The 0.84 wasn't. Full writeup + repo: [paper link] / [repo link].

**Employer-facing summary (3 sentences):** I built a client-holdout-validated ranking model on FlyRank's real search-analytics warehouse that prioritizes which content pages a limited-capacity editorial team should review first, beating a transparent rule baseline that turned out not to fire at all on held-out data. Along the way I found and fixed a genuine point-in-time data leak (costing 81% of the naive candidate rows) and a real reproducibility bug in a remote-query pipeline, and I deliberately proved -- not just claimed -- that an honest validation split matters, by showing the same model score 20 points lower under proper client-grouping than under a naive random split. The full pipeline, every bug found, and the final ranked recommendations are reproducible from the public repo.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
